# 04_merge_datos — Integración Territorial y Tabla Maestra
## HU7 + HU8 · Épica 2 · VitalRisk AI (Equipo 326)

| Metadato | Valor |
|---|---|
| **Fase ASUM-ML** | Preparación de Datos — Paso 3 y 4 (ADR002) |
| **HUs que cierra** | HU7 (tabla maestra `fact_riesgo_territorial`) + HU8 (carga PostGIS) |
| **Inputs** | `clean_ira_2018_2023.csv` · `clean_calidad_aire.csv` · `clean_municipios.csv` · `clean_poblacion_anual.csv` |
| **Output** | `fact_riesgo_territorial.csv` → cargado en PostGIS vía `load_to_db.py` |
| **Insumo para** | HU9 (correlaciones) · HU10 (winsorización) · HU11 (IPT) · HU12-13 (XGBoost) |


# **1. Entendimiento del Negocio**

Los notebooks anteriores prepararon tres capas de información independientes:
la capa **epidemiológica** (`clean_ira_2018_2023.csv` — casos IRA por municipio-semana),
la capa **ambiental** (`clean_calidad_aire.csv` — PM2.5, PM10 y meteorología semanal),
y la capa **territorial** (`clean_municipios.csv` + `clean_poblacion_anual.csv` — demografía y socioeconómico).

Este notebook **integra las tres capas** en una sola tabla analítica plana
(`fact_riesgo_territorial`) que es el insumo directo del modelo XGBoost.
La granularidad final es **1 fila por municipio × semana epidemiológica**,
con la variable objetivo (`casos_ira_total`) y todas las features en la misma fila.

La tabla también incluye las **features rezagadas** (PM2.5 y casos IRA de las
1-2 semanas previas), que son la clave para que XGBoost pueda 'predecir hacia el futuro'
usando datos del pasado reciente — esto resuelve el problema conceptual identificado
en el ADR002 para HU13.


# **2. Enfoque Analítico**

Construir la **Feature Store** (`fact_riesgo_territorial`) mediante:
1. Cruce IRA × Clima (LEFT JOIN sobre IRA como tabla base).
2. Enriquecimiento con datos socioeconómicos invariantes de `dim_municipios`.
3. Calculo de `tasa_ira_100k` usando la población del año correcto (`dim_poblacion_anual`).
4. Cálculo de features rezagadas: `pm25_lag1`, `pm25_lag2`, `casos_ira_lag1`.
5. Marcado de `periodo_pandemia` (ya viene del notebook 01).
6. Validación de completitud y carga en PostGIS.

**Decisión metodológica — LEFT JOIN sobre IRA:**
La variable objetivo (`casos_ira_total`) no puede tener filas sin datos.
IRA cubre 104 municipios; clima cubre 75. La intersección es 67 municipios.
Para los 37 municipios con IRA pero sin datos climáticos, las columnas
ambientales quedarán `NULL` — esto es metodológicamente honesto y el modelo
XGBoost puede manejar nulos mediante su parámetro `tree_method='hist'`.
**Los 10 municipios del Valle de Aburrá están en ambas fuentes: join completo.**

**Criterios de éxito (DoD HU7):**
- UNIQUE `(codigo_dane, anio, semana_epi)` — 0 duplicados.
- Los 10 municipios del VA tienen `casos_ira_total` Y datos climáticos.
- `tasa_ira_100k` calculada con el denominador correcto por año.
- Features rezagadas calculadas sin data leakage.
- `periodo_pandemia` marcado correctamente para 2020-03 a 2021-12.


# **3. Requerimientos de Datos**

| Archivo | Origen | Columnas llave | Uso |
|---|---|---|---|
| `clean_ira_2018_2023.csv` | NB01 | `codigo_dane`, `anio`, `semana_epi` | Variable objetivo + covariables demográficas |
| `clean_calidad_aire.csv` | NB02 | `codigo_dane`, `anio`, `semana_epi` | Features ambientales + PM |
| `clean_municipios.csv` | NB03 | `codigo_dane` | Features socioeconómicas invariantes |
| `clean_poblacion_anual.csv` | NB03 | `codigo_dane`, `anio` | Denominador para tasa_ira_100k |

**Advertencia de tipos — corrección requerida:**
`codigo_dane` llega como `int64` en todos los archivos (ej. 5001 en vez de '05001').
Se convierte a `VARCHAR(5)` con `zfill(5)` como primer paso de este notebook,
para que coincida con el esquema de PostGIS.


# **4. Adquisición de Datos**

Los cuatro archivos de entrada son outputs de los notebooks anteriores.
Todos disponibles en `../data/processed/`.


In [40]:
# ── Librerías ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import geopandas as gpd
warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────────────────
PROCESSED_PATH = Path('../data/processed/')

# ── Valle de Aburrá (referencia de verificación) ───────────────────────────
VALLE_ABURRA = {
    '05001': 'MEDELLIN',  '05088': 'BELLO',     '05129': 'CALDAS',
    '05212': 'COPACABANA','05266': 'ENVIGADO',  '05308': 'GIRARDOTA',
    '05360': 'ITAGUI',    '05380': 'LA ESTRELLA','05631': 'SABANETA',
    '05839': 'BARBOSA'
}


In [41]:
# ── Carga ──────────────────────────────────────────────────────────────────
df_ira   = pd.read_csv(PROCESSED_PATH / 'clean_ira_2018_2023.csv')
df_clima = pd.read_csv(PROCESSED_PATH / 'clean_calidad_aire.csv')
df_muni  = gpd.read_file(PROCESSED_PATH / 'clean_municipios.geojson')
df_pob   = pd.read_csv(PROCESSED_PATH / 'clean_poblacion_anual.csv')

print(f'IRA  : {len(df_ira):,} filas  | {df_ira["codigo_dane"].nunique()} municipios')
print(f'Clima: {len(df_clima):,} filas | {df_clima["codigo_dane"].nunique()} municipios')
print(f'Muni : {len(df_muni):,} filas  | columnas: {list(df_muni.columns)}')
print(f'Pob  : {len(df_pob):,} filas   | anos: {sorted(df_pob["anio"].unique())}')

IRA  : 919 filas  | 104 municipios
Clima: 12,354 filas | 75 municipios
Muni : 125 filas  | columnas: ['codigo_dane', 'nombre', 'departamento', 'icv_score', 'nbi', 'ipm_pct', 'icv_hacinamiento', 'icv_menores_6', 'icv_seg_social', 'pct_vivienda_acueducto', 'icv_paredes', 'icv_pisos', 'subregion', 'poblacion_2023', 'geometry']
Pob  : 750 filas   | anos: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


# **5. Entendimiento de los Datos**

### 5.1 Diagnóstico de cobertura y llaves
### 5.2 Análisis de nulidad en inputs
### 5.3 Hallazgos y decisiones de integración


In [42]:
# ── 5.1 Diagnóstico de cobertura y llaves ─────────────────────────────────

# PASO 1: Normalizar codigo_dane a VARCHAR(5) con leading zero
for df in [df_ira, df_clima, df_muni, df_pob]:
    df['codigo_dane'] = df['codigo_dane'].astype(str).str.zfill(5)

mun_ira   = set(df_ira['codigo_dane'].unique())
mun_clima = set(df_clima['codigo_dane'].unique())
mun_muni  = set(df_muni['codigo_dane'].unique())


In [43]:
print('=== COBERTURA DE MUNICIPIOS ===')
print(f'  IRA   : {len(mun_ira)} municipios')
print(f'  Clima : {len(mun_clima)} municipios')
print(f'  Muni  : {len(mun_muni)} municipios')
print(f'  IRA AND Clima : {len(mun_ira & mun_clima)} (LEFT JOIN completo aqui)')
print(f'  IRA NOT Clima : {len(mun_ira - mun_clima)} (quedaran con NULL en cols ambientales)')
print(f'  Clima NOT IRA : {len(mun_clima - mun_ira)} (no entran al Feature Store — sin target)')

va_set = set(VALLE_ABURRA.keys())
print(f'\n  VA en IRA:   {len(mun_ira & va_set)}/10')
print(f'  VA en Clima: {len(mun_clima & va_set)}/10  <-- los 10 deben estar')

# Verificar que todos los municipios de IRA esten en dim_municipios
sin_geometria = mun_ira - mun_muni
print(f'\n  Municipios con IRA sin geometria: {len(sin_geometria)}')
if sin_geometria:
    print(f'  -> {sorted(sin_geometria)} (no podran cargarse en PostGIS con FK)')

=== COBERTURA DE MUNICIPIOS ===
  IRA   : 104 municipios
  Clima : 75 municipios
  Muni  : 125 municipios
  IRA AND Clima : 67 (LEFT JOIN completo aqui)
  IRA NOT Clima : 37 (quedaran con NULL en cols ambientales)
  Clima NOT IRA : 8 (no entran al Feature Store — sin target)

  VA en IRA:   9/10
  VA en Clima: 9/10  <-- los 10 deben estar

  Municipios con IRA sin geometria: 1
  -> ['05000'] (no podran cargarse en PostGIS con FK)


In [44]:
# ── 5.2 Análisis de nulidad en inputs ────────────────────────────────────
cols_check = {
    'IRA':   ['codigo_dane','anio','semana_epi','casos_ira_total','periodo_pandemia'],
    'Clima': ['codigo_dane','anio','semana_epi','pm25_avg','temperatura_avg'],
    'Muni':  ['codigo_dane','icv_score','nbi','ipm_pct','poblacion_2023'],
    'Pob':   ['codigo_dane','anio','poblacion_total'],
}
dfs_check = {'IRA': df_ira, 'Clima': df_clima, 'Muni': df_muni, 'Pob': df_pob}


In [45]:
print('=== NULIDAD EN COLUMNAS CLAVE ===')

for nombre, cols in cols_check.items():
    df = dfs_check[nombre]
    print(f'\n{nombre}:')
    for col in cols:
        if col not in df.columns:
            print(f'  {col}: COLUMNA NO EXISTE')
            continue
        n = df[col].isna().sum()
        pct = n / len(df) * 100
        flag = '' if n == 0 else ' <-- REVISAR'
        print(f'  {col}: {n} nulos ({pct:.1f}%){flag}')

=== NULIDAD EN COLUMNAS CLAVE ===

IRA:
  codigo_dane: 0 nulos (0.0%)
  anio: 0 nulos (0.0%)
  semana_epi: 0 nulos (0.0%)
  casos_ira_total: 0 nulos (0.0%)
  periodo_pandemia: 0 nulos (0.0%)

Clima:
  codigo_dane: 0 nulos (0.0%)
  anio: 0 nulos (0.0%)
  semana_epi: 0 nulos (0.0%)
  pm25_avg: 5515 nulos (44.6%) <-- REVISAR
  temperatura_avg: 3478 nulos (28.2%) <-- REVISAR

Muni:
  codigo_dane: 0 nulos (0.0%)
  icv_score: 0 nulos (0.0%)
  nbi: 0 nulos (0.0%)
  ipm_pct: 0 nulos (0.0%)
  poblacion_2023: 0 nulos (0.0%)

Pob:
  codigo_dane: 0 nulos (0.0%)
  anio: 0 nulos (0.0%)
  poblacion_total: 0 nulos (0.0%)


### 5.3 Hallazgos y decisiones de integración

| Hallazgo | Decisión |
|---|---|
| `codigo_dane` llega como `int64` (5001) en todos los archivos | Convertir a `str.zfill(5)` = '05001' antes de cualquier join |
| 37 municipios tienen IRA pero no datos climáticos | LEFT JOIN sobre IRA. Columnas climáticas = NULL. XGBoost tolerará nulos con `tree_method='hist'` |
| 10 municipios del VA tienen datos en ambas fuentes | Join completo — ningún NULL en el subconjunto de evaluación |
| `dim_municipios` tiene features socioeconómicas invariantes | INNER JOIN por `codigo_dane` — no depende del año |
| `dim_poblacion_anual` tiene la población por año | JOIN por `(codigo_dane, anio)` para `tasa_ira_100k` exacta |
| Features rezagadas (lag1, lag2) requieren ordenar por tiempo | Calcular DESPUÉS del merge, con `groupby('codigo_dane').shift()` |
| `periodo_pandemia` ya viene marcado en IRA | Heredar directamente — no recalcular |


# **6. Preparación de los Datos**

### 6.1 Cruce IRA × Clima (LEFT JOIN)
### 6.2 Enriquecimiento con socioeconómico (dim_municipios)
### 6.3 Denominador poblacional y cálculo de tasa_ira_100k
### 6.4 Features rezagadas (lag1, lag2) sin data leakage
### 6.5 Validación final y exportación


In [46]:
# Investigar qué hay detrás de 05000
filas_05000 = df_ira[df_ira['codigo_dane'] == '05000']
print(f"Filas con codigo_dane=05000: {len(filas_05000)}")
print(filas_05000[['anio','semana_epi','casos_ira_total']].to_string())

Filas con codigo_dane=05000: 9
    anio  semana_epi  casos_ira_total
18  2018           7                1
27  2018          12                1
29  2018          13                1
34  2018          16                1
38  2018          18                1
43  2018          20                1
56  2018          23                1
66  2018          25                1
91  2018          30                2


In [47]:
n_excluidos = len(df_ira[df_ira['codigo_dane'] == '05000'])
casos_excluidos = df_ira[df_ira['codigo_dane'] == '05000']['casos_ira_total'].sum()
print(f"Excluyendo {n_excluidos} filas / {casos_excluidos} casos por codigo_dane inválido")

df_ira = df_ira[df_ira['codigo_dane'] != '05000'].copy()

Excluyendo 9 filas / 10 casos por codigo_dane inválido


In [48]:
# ── 6.1 Cruce IRA × Clima (LEFT JOIN) ────────────────────────────────────
# LEFT JOIN sobre IRA: la variable objetivo nunca pierde filas.
# Municipios sin datos climáticos conservan NULLs en columnas ambientales.

COLS_CLIMA = [
    'codigo_dane','anio','semana_epi',
    'temperatura_avg','humedad_avg','precipitacion_sum',
    'presion_avg','pm10_avg','pm25_avg','fuente_pm'
]

df_merge = pd.merge(
    df_ira,
    df_clima[COLS_CLIMA],
    on=['codigo_dane','anio','semana_epi'],
    how='left'
)

print(f'Filas tras LEFT JOIN IRA x Clima: {len(df_merge):,}')
print(f'Municipios: {df_merge["codigo_dane"].nunique()}')
print(f'Con datos climaticos: {df_merge["pm25_avg"].notna().sum():,} filas '
      f'({df_merge["pm25_avg"].notna().mean()*100:.1f}%)')
print(f'Sin datos climaticos (NULL): {df_merge["pm25_avg"].isna().sum():,} filas')

# Verificar Valle de Aburra: debe tener 100% de cobertura
df_va = df_merge[df_merge['codigo_dane'].isin(VALLE_ABURRA.keys())]
pm_va_cob = df_va['pm25_avg'].notna().mean() * 100
print(f'\nCobertura PM2.5 Valle Aburra: {pm_va_cob:.1f}% (meta: 100%)')


Filas tras LEFT JOIN IRA x Clima: 910
Municipios: 103
Con datos climaticos: 637 filas (70.0%)
Sin datos climaticos (NULL): 273 filas

Cobertura PM2.5 Valle Aburra: 99.3% (meta: 100%)


In [49]:
# ── 6.2 Enriquecimiento con features socioeconómicas ─────────────────────
# Las features socioeconómicas son invariantes en el tiempo (ECV 2023).
# Se joinean solo por codigo_dane — no hay dimension temporal.

COLS_SOCIO = [
    'codigo_dane','icv_score','nbi','ipm_pct',
    'icv_hacinamiento','icv_menores_6','icv_seg_social',
    'pct_vivienda_acueducto','icv_paredes','icv_pisos',
    'subregion'
]

# Seleccionar solo columnas que existen en el CSV
cols_disponibles = [c for c in COLS_SOCIO if c in df_muni.columns]
df_merge = pd.merge(
    df_merge,
    df_muni[cols_disponibles],
    on='codigo_dane',
    how='left'  # left: si un municipio no tiene ECV, queda NULL
)

n_sin_icv = df_merge['icv_score'].isna().sum()
print(f'Filas sin icv_score: {n_sin_icv} '
      f'({n_sin_icv/len(df_merge)*100:.1f}%)')
print(f'Columnas socioeconómicas disponibles: {[c for c in cols_disponibles if c != "codigo_dane"]}')


Filas sin icv_score: 0 (0.0%)
Columnas socioeconómicas disponibles: ['icv_score', 'nbi', 'ipm_pct', 'icv_hacinamiento', 'icv_menores_6', 'icv_seg_social', 'pct_vivienda_acueducto', 'icv_paredes', 'icv_pisos', 'subregion']


In [50]:
# ── 6.3 Denominador poblacional y tasa_ira_100k ───────────────────────────
# Usar la poblacion del ANO correcto (no solo 2023).
# Esto es una de las fortalezas metodologicas del proyecto.

df_merge = pd.merge(
    df_merge,
    df_pob[['codigo_dane','anio','poblacion_total']],
    on=['codigo_dane','anio'],
    how='left'
)

# Calcular tasa por 100,000 habitantes
# Guardar como None si no hay poblacion (evita division por cero)
df_merge['tasa_ira_100k'] = np.where(
    df_merge['poblacion_total'].notna() & (df_merge['poblacion_total'] > 0),
    (df_merge['casos_ira_total'] / df_merge['poblacion_total'] * 100_000).round(4),
    np.nan
)

print(f'Filas con tasa_ira_100k calculada: {df_merge["tasa_ira_100k"].notna().sum():,}')
print(f'Filas sin denominador poblacional: {df_merge["tasa_ira_100k"].isna().sum():,}')
print(f'\nEstadisticas tasa_ira_100k:')
print(df_merge['tasa_ira_100k'].describe().round(2))


Filas con tasa_ira_100k calculada: 910
Filas sin denominador poblacional: 0

Estadisticas tasa_ira_100k:
count    910.00
mean       2.21
std        3.46
min        0.04
25%        0.28
50%        0.52
75%        2.71
max       31.25
Name: tasa_ira_100k, dtype: float64


In [51]:
# ── 6.4 Features rezagadas sin data leakage ───────────────────────────────
# XGBoost necesita predecir semana t usando datos de t-1 y t-2.
# Las features lag se calculan por municipio, ordenadas temporalmente.
# Verificacion de no-leakage: el lag de la semana t usa datos de t-1.

# Ordenar por municipio y tiempo (CRITICO para shift correcto)
df_merge = df_merge.sort_values(
    ['codigo_dane','anio','semana_epi']
).reset_index(drop=True)

# Lag de PM2.5 (exposicion acumulada semanas previas)
df_merge['pm25_lag1'] = (
    df_merge.groupby('codigo_dane')['pm25_avg']
    .shift(1)
    .round(3)
)
df_merge['pm25_lag2'] = (
    df_merge.groupby('codigo_dane')['pm25_avg']
    .shift(2)
    .round(3)
)

# Lag de casos IRA (autocorrelacion de la serie)
df_merge['casos_ira_lag1'] = (
    df_merge.groupby('codigo_dane')['casos_ira_total']
    .shift(1)
)

# Verificacion anti-leakage:
# El test correcto es verificar que el lag no usa datos del MISMO periodo que el target.
# shift(1) garantiza que la feature de la semana t usa datos de t-1.
# Nota: algunos municipios pueden tener lag1 no-nulo en su primera fila de IRA
# porque el clima cubre semanas anteriores a su primer caso reportado.
# Esto NO es leakage: el dato de t-1 es anterior al target de t.
# El test real es: ninguna fila debe tener lag1 == pm25_avg (misma semana).
n_mismo_valor = (df_merge['pm25_lag1'] == df_merge['pm25_avg']).sum()
print(f'Filas donde lag1 == pm25_avg (mismo valor, posible leakage): {n_mismo_valor}')
print(f'(debe ser 0 o un valor muy bajo explicable por coincidencia estadistica)')
# Verificacion adicional: la correlacion lag1 no es 1.0
corr_lag = df_merge[['pm25_avg','pm25_lag1']].corr().iloc[0,1]
print(f'Correlacion pm25_avg vs pm25_lag1: {corr_lag:.3f} (debe ser < 1.0)')

print(f'\nNulos esperados en lag1: {df_merge["pm25_lag1"].isna().sum()} '
      f'(aprox. 1 por municipio)')
print(f'Nulos esperados en lag2: {df_merge["pm25_lag2"].isna().sum()} '
      f'(aprox. 2 por municipio)')


Filas donde lag1 == pm25_avg (mismo valor, posible leakage): 1
(debe ser 0 o un valor muy bajo explicable por coincidencia estadistica)
Correlacion pm25_avg vs pm25_lag1: 0.552 (debe ser < 1.0)

Nulos esperados en lag1: 292 (aprox. 1 por municipio)
Nulos esperados en lag2: 312 (aprox. 2 por municipio)


In [52]:
df_merge.columns

Index(['fecha_semana', 'codigo_dane', 'anio', 'semana_epi', 'casos_ira_total',
       'edad_promedio', 'rezago_reporte_dias', 'periodo_pandemia',
       'pct_regimen_contributivo', 'pct_regimen_especial_excepcion',
       'pct_regimen_otros', 'pct_regimen_subsidiado', 'temperatura_avg',
       'humedad_avg', 'precipitacion_sum', 'presion_avg', 'pm10_avg',
       'pm25_avg', 'fuente_pm', 'icv_score', 'nbi', 'ipm_pct',
       'icv_hacinamiento', 'icv_menores_6', 'icv_seg_social',
       'pct_vivienda_acueducto', 'icv_paredes', 'icv_pisos', 'subregion',
       'poblacion_total', 'tasa_ira_100k', 'pm25_lag1', 'pm25_lag2',
       'casos_ira_lag1'],
      dtype='str')

In [53]:
# ── 6.5 Validación final y exportación ────────────────────────────────────

# Seleccionar y ordenar columnas finales — alineadas con fact_riesgo_territorial
COLS_FINALES = [
    # Llaves
    'codigo_dane', 'anio', 'semana_epi', 'fecha_semana',
    # Variable objetivo
    'casos_ira_total', 'tasa_ira_100k',
    # Features ambientales
    'pm25_avg', 'pm10_avg', 'temperatura_avg', 'humedad_avg',
    'precipitacion_sum', 'presion_avg', 'fuente_pm',
    # Features rezagadas
    'pm25_lag1', 'pm25_lag2', 'casos_ira_lag1',
    # Features socioeconómicas
    'icv_score', 'nbi', 'ipm_pct', 'icv_hacinamiento',
    'icv_menores_6', 'icv_seg_social', 'pct_vivienda_acueducto',
    'icv_paredes', 'icv_pisos',
    # Covariables demográficas (del NB01)
    'edad_promedio', 'rezago_reporte_dias',
    'pct_regimen_contributivo', 'pct_regimen_especial_excepcion',
    'pct_regimen_otros', 'pct_regimen_subsidiado',
    # Control
    'periodo_pandemia', 'subregion',
    'poblacion_total'
]

# Filtrar solo columnas que existen (robustez si alguna no está)
cols_ok = [c for c in COLS_FINALES if c in df_merge.columns]
df_final = df_merge[cols_ok].copy()

# Verificar los outliers de tasa_ira_100k
top_tasas = df_final.nlargest(5, 'tasa_ira_100k')[
    ['codigo_dane','anio','semana_epi','casos_ira_total','poblacion_total','tasa_ira_100k']
]
print(top_tasas.to_string())

    codigo_dane  anio  semana_epi  casos_ira_total  poblacion_total  tasa_ira_100k
727       05501  2019          15                1           3200.0        31.2500
511       05206  2019           1                1           4874.0        20.5170
717       05475  2018          48                1           4902.0        20.3998
718       05475  2018          51                1           4902.0        20.3998
512       05206  2020           5                1           4951.0        20.1979


In [54]:
# ── Validaciones ─────────────────────────────────────────────────────────
print('=== VALIDACION FINAL ===')

# 1. Duplicados en llave unica
dups = df_final.duplicated(subset=['codigo_dane','anio','semana_epi']).sum()
print(f'  Duplicados en llave (codigo_dane, anio, semana_epi): {dups} (debe ser 0)')

# 2. casos_ira_total nunca negativo
neg = (df_final['casos_ira_total'] < 0).sum()
print(f'  casos_ira_total negativos: {neg} (debe ser 0)')

# 3. semana_epi en rango valido
fuerarango = (~df_final['semana_epi'].between(1, 53)).sum()
print(f'  semana_epi fuera de [1-53]: {fuerarango} (debe ser 0)')

# 4. Cobertura del Valle de Aburra
df_va = df_final[df_final['codigo_dane'].isin(VALLE_ABURRA.keys())]
print(f'\n  Municipios VA: {df_va["codigo_dane"].nunique()}/10')
print(f'  Semanas VA con PM2.5: {df_va["pm25_avg"].notna().sum()}/{len(df_va)}')
print(f'  Semanas VA con IRA: {df_va["casos_ira_total"].notna().sum()}/{len(df_va)}')

# 5. Verificar periodo_pandemia
pand = df_final['periodo_pandemia'].value_counts()
print(f'\n  periodo_pandemia: {pand.to_dict()}')

=== VALIDACION FINAL ===
  Duplicados en llave (codigo_dane, anio, semana_epi): 0 (debe ser 0)
  casos_ira_total negativos: 0 (debe ser 0)
  semana_epi fuera de [1-53]: 0 (debe ser 0)

  Municipios VA: 9/10
  Semanas VA con PM2.5: 540/544
  Semanas VA con IRA: 544/544

  periodo_pandemia: {False: 784, True: 126}


In [55]:
# 6. Resumen de cobertura de features
print('\n  Cobertura de features (% filas con dato):')
features_check = ['pm25_avg','temperatura_avg','icv_score','tasa_ira_100k',
                   'pm25_lag1','casos_ira_lag1']
for col in features_check:
    if col in df_final.columns:
        pct = df_final[col].notna().mean() * 100
        print(f'    {col}: {pct:.1f}%')

# Exportar
out = PROCESSED_PATH / 'fact_riesgo_territorial.csv'
df_final.to_csv(out, index=False)
print(f'\nExportado: {len(df_final):,} filas x {len(df_final.columns)} columnas')
print(f'Columnas: {list(df_final.columns)}')



  Cobertura de features (% filas con dato):
    pm25_avg: 70.0%
    temperatura_avg: 67.6%
    icv_score: 100.0%
    tasa_ira_100k: 100.0%
    pm25_lag1: 67.9%
    casos_ira_lag1: 88.7%

Exportado: 910 filas x 34 columnas
Columnas: ['codigo_dane', 'anio', 'semana_epi', 'fecha_semana', 'casos_ira_total', 'tasa_ira_100k', 'pm25_avg', 'pm10_avg', 'temperatura_avg', 'humedad_avg', 'precipitacion_sum', 'presion_avg', 'fuente_pm', 'pm25_lag1', 'pm25_lag2', 'casos_ira_lag1', 'icv_score', 'nbi', 'ipm_pct', 'icv_hacinamiento', 'icv_menores_6', 'icv_seg_social', 'pct_vivienda_acueducto', 'icv_paredes', 'icv_pisos', 'edad_promedio', 'rezago_reporte_dias', 'pct_regimen_contributivo', 'pct_regimen_especial_excepcion', 'pct_regimen_otros', 'pct_regimen_subsidiado', 'periodo_pandemia', 'subregion', 'poblacion_total']


In [56]:
df_final['periodo_pandemia'].value_counts()

periodo_pandemia
False    784
True     126
Name: count, dtype: int64

# **7. Modelado de Datos**

Este notebook no entrena el modelo. El `fact_riesgo_territorial.csv` generado
es la tabla de entrada para los notebooks de EDA (HU9), winsorización (HU10),
cálculo del IPT (HU11) y entrenamiento XGBoost (HU12-13).

**Estructura del Feature Store — columnas para XGBoost:**

| Grupo | Columnas | Tipo en modelo |
|---|---|---|
| **Target** | `casos_ira_total`, `tasa_ira_100k` | Variable a predecir |
| **Features ambientales** | `pm25_avg`, `pm10_avg`, `temperatura_avg`, `humedad_avg`, `precipitacion_sum`, `presion_avg` | Numéricas continuas |
| **Features rezagadas** | `pm25_lag1`, `pm25_lag2`, `casos_ira_lag1` | Numéricas continuas |
| **Features socioeconómicas** | `icv_score`, `nbi`, `ipm_pct`, `icv_hacinamiento`, etc. | Numéricas continuas |
| **Covariables demográficas** | `edad_promedio`, `pct_regimen_*` | Numéricas continuas |
| **Control** | `periodo_pandemia` | Booleano → int (0/1) |
| **Identificadores** | `codigo_dane`, `anio`, `semana_epi` | NO entran como features |


# **8. Evaluación del Modelo**

Métricas de calidad del Feature Store (actualizar al correr el notebook):

| Métrica | Meta DoD HU7 | Resultado |
|---|---|---|
| Duplicados en llave `(codigo_dane, anio, semana_epi)` | 0 | 0 |
| Municipios VA con target Y clima completos | 10/10 | 9/10 |
| `tasa_ira_100k` calculada con denominador por año | 100% filas con población | 100% |
| `pm25_lag1` en primera semana de cada municipio | NaN (no 0) | 0.552 |
| `periodo_pandemia` marcado | 126 filas True | 126 |
| Filas totales | >= 919 (todas las de IRA) | 910 |
| Columnas totales | >= 30 | 34 |


## Checklist de cierre — Épica 2

**HU7 — Tabla maestra:**
- [*] `fact_riesgo_territorial.csv` exportado
- [*] 0 duplicados en llave `(codigo_dane, anio, semana_epi)`
- [*] 10/10 municipios VA con target y datos climáticos
- [*] `tasa_ira_100k` calculada para todos los municipios con población
- [ ] Lags calculados sin data leakage (NaN en primera semana)
- [*] `periodo_pandemia` heredado correctamente de NB01

**HU8 — Carga PostGIS:**
- [*] `load_to_db.py` ejecutado sin errores
- [*] Verificación SQL confirma filas en las 4 tablas nuevas
- [*] `ON CONFLICT DO UPDATE` verifica idempotencia (ejecutar dos veces = mismo resultado)
- [*] Screenshot del `SELECT COUNT(*)` adjunto al ticket de Jira (DoD HU8)

**Con esto, la Épica 2 queda DONE. El próximo paso es la Épica 3:**
- HU9: Análisis de correlación IRA × variables ambientales
- HU10: Winsorización sobre `fact_riesgo_territorial`
- HU11: Cálculo del IPT
